# Embedding & Model Diagnostics

This notebook diagnoses why recommendation metrics are poor:
- **Data Quality Analysis**: Sparsity, distribution, cold start issues
- **Embedding Quality**: Visual embedding analysis, learned embedding analysis
- **Training Diagnostics**: Overfitting analysis, learning dynamics
- **Baseline Comparisons**: Random, popularity, content-based
- **SOTA API Comparison**: Test against external recommender APIs

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import torch
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Imports successful!")

## 1. Load Data and Embeddings

In [ ]:
from src.data.processor import DataProcessor
from src.utils.config import load_config

# Load config and data
config = load_config()
processor = DataProcessor(config)

# Load raw data
items_df, events_df = processor.load_data()
interactions_df = processor.build_interactions()
items_df, interactions_df = processor.encode_features()

print(f"Items: {len(items_df):,}")
print(f"Users: {interactions_df['user_idx'].nunique():,}")
print(f"Interactions: {len(interactions_df):,}")

In [ ]:
# Load embeddings
import os

visual_embeddings = None
visual_emb_path = 'data/embeddings/embeddings.npy'
if os.path.exists(visual_emb_path):
    visual_embeddings = np.load(visual_emb_path)
    print(f"Visual embeddings shape: {visual_embeddings.shape}")
else:
    print("No visual embeddings found")

# Load trained model embeddings if available
two_tower_embeddings = None
tt_emb_path = 'checkpoints/two_tower_index/embeddings.npy'
if os.path.exists(tt_emb_path):
    two_tower_embeddings = np.load(tt_emb_path)
    print(f"Two-tower embeddings shape: {two_tower_embeddings.shape}")

## 2. Data Sparsity Analysis

Key metrics to check:
- User-item matrix density
- User interaction distribution (cold start problem)
- Item popularity distribution (long tail problem)

In [ ]:
# Compute sparsity metrics
num_users = interactions_df['user_idx'].nunique()
num_items = len(items_df)
num_interactions = len(interactions_df)

density = num_interactions / (num_users * num_items) * 100
avg_interactions_per_user = num_interactions / num_users
avg_interactions_per_item = num_interactions / num_items

print("=" * 50)
print("DATA SPARSITY ANALYSIS")
print("=" * 50)
print(f"Matrix size: {num_users:,} users × {num_items:,} items = {num_users * num_items:,} cells")
print(f"Observed interactions: {num_interactions:,}")
print(f"Matrix density: {density:.4f}%")
print(f"Avg interactions/user: {avg_interactions_per_user:.1f}")
print(f"Avg interactions/item: {avg_interactions_per_item:.1f}")
print()
print("⚠️ DIAGNOSIS:")
if density < 0.1:
    print(f"  - VERY SPARSE data ({density:.4f}%). Need strong regularization or content features.")
if avg_interactions_per_user < 10:
    print(f"  - Cold start problem: avg {avg_interactions_per_user:.1f} interactions/user is low.")
if avg_interactions_per_item < 5:
    print(f"  - Many items have few interactions ({avg_interactions_per_item:.1f}/item avg).")

In [ ]:
# User interaction distribution
user_counts = interactions_df.groupby('user_idx').size()
item_counts = interactions_df.groupby('item_idx').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# User distribution
axes[0].hist(user_counts, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Interactions')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('User Interaction Distribution')
axes[0].axvline(user_counts.median(), color='red', linestyle='--', label=f'Median: {user_counts.median():.0f}')
axes[0].axvline(user_counts.mean(), color='green', linestyle='--', label=f'Mean: {user_counts.mean():.1f}')
axes[0].legend()

# Item distribution (log scale)
axes[1].hist(item_counts, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of Interactions')
axes[1].set_ylabel('Number of Items')
axes[1].set_title('Item Popularity Distribution (Long Tail)')
axes[1].set_yscale('log')
axes[1].axvline(item_counts.median(), color='red', linestyle='--', label=f'Median: {item_counts.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

# Cold start analysis
cold_users = (user_counts <= 3).sum()
cold_items = (item_counts <= 1).sum()
print(f"\nCold Start Analysis:")
print(f"  - Users with ≤3 interactions: {cold_users:,} ({cold_users/num_users*100:.1f}%)")
print(f"  - Items with ≤1 interaction: {cold_items:,} ({cold_items/num_items*100:.1f}%)")

## 3. Visual Embedding Quality Analysis

In [ ]:
if visual_embeddings is not None:
    print("=" * 50)
    print("VISUAL EMBEDDING ANALYSIS")
    print("=" * 50)
    
    # Basic stats
    print(f"Shape: {visual_embeddings.shape}")
    print(f"Dtype: {visual_embeddings.dtype}")
    print(f"Min: {visual_embeddings.min():.4f}")
    print(f"Max: {visual_embeddings.max():.4f}")
    print(f"Mean: {visual_embeddings.mean():.4f}")
    print(f"Std: {visual_embeddings.std():.4f}")
    
    # Check for NaN/Inf
    nan_count = np.isnan(visual_embeddings).sum()
    inf_count = np.isinf(visual_embeddings).sum()
    print(f"NaN values: {nan_count}")
    print(f"Inf values: {inf_count}")
    
    # Check L2 norms (should be ~1 for normalized embeddings)
    norms = np.linalg.norm(visual_embeddings, axis=1)
    print(f"\nL2 Norms: min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
    
    if norms.std() > 0.1:
        print("⚠️ Embeddings NOT normalized! Consider L2 normalization.")
else:
    print("No visual embeddings to analyze")

In [ ]:
if visual_embeddings is not None:
    # Sample for visualization (too many points = slow)
    n_sample = min(5000, len(visual_embeddings))
    sample_idx = np.random.choice(len(visual_embeddings), n_sample, replace=False)
    sample_emb = visual_embeddings[sample_idx]
    
    # Get categories for coloring
    sample_items = items_df.iloc[sample_idx]
    
    # PCA visualization
    print("Computing PCA...")
    pca = PCA(n_components=2)
    emb_pca = pca.fit_transform(sample_emb)
    
    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Color by top categories
    top_categories = sample_items['category_name'].value_counts().head(10).index
    colors = sample_items['category_name'].apply(lambda x: x if x in top_categories else 'Other')
    
    scatter = axes[0].scatter(emb_pca[:, 0], emb_pca[:, 1], c=pd.factorize(colors)[0], 
                              alpha=0.5, s=10, cmap='tab10')
    axes[0].set_title('Visual Embeddings (PCA) - Colored by Category')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    
    # t-SNE (slower but better for clusters)
    print("Computing t-SNE...")
    tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42)
    emb_tsne = tsne.fit_transform(sample_emb)
    
    scatter = axes[1].scatter(emb_tsne[:, 0], emb_tsne[:, 1], c=pd.factorize(colors)[0], 
                              alpha=0.5, s=10, cmap='tab10')
    axes[1].set_title('Visual Embeddings (t-SNE) - Colored by Category')
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️ DIAGNOSIS:")
    print("  - If categories form distinct clusters: Visual embeddings are good!")
    print("  - If it's a blob: Visual embeddings don't capture category semantics.")

In [ ]:
if visual_embeddings is not None:
    # Test embedding quality: Same-category items should be more similar
    print("\nEmbedding Quality Test: Category Coherence")
    print("-" * 50)
    
    # Get category assignments
    categories = items_df['category_name'].values
    unique_cats = items_df['category_name'].unique()
    
    # Sample 1000 pairs for within-category similarity
    within_sims = []
    between_sims = []
    
    for _ in range(2000):
        i, j = np.random.choice(len(visual_embeddings), 2, replace=False)
        sim = cosine_similarity([visual_embeddings[i]], [visual_embeddings[j]])[0, 0]
        
        if categories[i] == categories[j]:
            within_sims.append(sim)
        else:
            between_sims.append(sim)
    
    print(f"Within-category avg similarity: {np.mean(within_sims):.4f}")
    print(f"Between-category avg similarity: {np.mean(between_sims):.4f}")
    print(f"Difference (higher = better): {np.mean(within_sims) - np.mean(between_sims):.4f}")
    
    if np.mean(within_sims) - np.mean(between_sims) < 0.05:
        print("\n⚠️ WARNING: Embeddings don't distinguish categories well!")
        print("   Consider: Better image encoder, fine-tuning, or using text embeddings.")

## 4. Learned Embedding Analysis (Two-Tower)

In [ ]:
# Load the trained model if available
model = None
model_path = 'checkpoints/model.pt'

if os.path.exists(model_path):
    from src.models.two_tower import TwoTowerModel
    try:
        model = TwoTowerModel.load(model_path, device='cpu')
        print("Model loaded successfully!")
        print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    except Exception as e:
        print(f"Failed to load model: {e}")
else:
    print(f"No model checkpoint found at {model_path}")

In [ ]:
if model is not None:
    # Analyze user and item embedding spaces
    print("=" * 50)
    print("LEARNED EMBEDDING ANALYSIS")
    print("=" * 50)
    
    # Get user embeddings
    user_emb_weights = model.user_tower.user_embedding.weight.detach().cpu().numpy()
    item_emb_weights = model.item_tower.item_embedding.weight.detach().cpu().numpy()
    
    print(f"User embeddings shape: {user_emb_weights.shape}")
    print(f"Item embeddings shape: {item_emb_weights.shape}")
    
    # Check for collapse (all embeddings similar)
    user_norms = np.linalg.norm(user_emb_weights[1:], axis=1)  # Skip padding idx 0
    item_norms = np.linalg.norm(item_emb_weights[1:], axis=1)
    
    print(f"\nUser embedding norms: mean={user_norms.mean():.4f}, std={user_norms.std():.4f}")
    print(f"Item embedding norms: mean={item_norms.mean():.4f}, std={item_norms.std():.4f}")
    
    # Check cosine similarity distribution
    sample_users = user_emb_weights[np.random.choice(len(user_emb_weights)-1, 500, replace=False) + 1]
    sample_items = item_emb_weights[np.random.choice(len(item_emb_weights)-1, 500, replace=False) + 1]
    
    user_user_sim = cosine_similarity(sample_users)
    item_item_sim = cosine_similarity(sample_items)
    user_item_sim = cosine_similarity(sample_users, sample_items)
    
    # Get off-diagonal similarities
    uu_off = user_user_sim[np.triu_indices(500, k=1)]
    ii_off = item_item_sim[np.triu_indices(500, k=1)]
    ui_flat = user_item_sim.flatten()
    
    print(f"\nUser-User similarity: mean={uu_off.mean():.4f}, std={uu_off.std():.4f}")
    print(f"Item-Item similarity: mean={ii_off.mean():.4f}, std={ii_off.std():.4f}")
    print(f"User-Item similarity: mean={ui_flat.mean():.4f}, std={ui_flat.std():.4f}")
    
    if uu_off.std() < 0.1 or ii_off.std() < 0.1:
        print("\n⚠️ WARNING: Embedding collapse detected!")
        print("   All embeddings are too similar - model isn't learning.")

## 5. Training Dynamics Analysis

In [ ]:
# Check MLflow for training history
import sqlite3

mlflow_db = 'mlflow.db'
if os.path.exists(mlflow_db):
    conn = sqlite3.connect(mlflow_db)
    
    # Get latest run metrics
    metrics_df = pd.read_sql_query("""
        SELECT m.key, m.value, m.step 
        FROM metrics m
        JOIN runs r ON m.run_uuid = r.run_uuid
        ORDER BY r.start_time DESC, m.step
        LIMIT 1000
    """, conn)
    
    if len(metrics_df) > 0:
        print("=" * 50)
        print("TRAINING HISTORY ANALYSIS")
        print("=" * 50)
        
        # Plot training curves
        train_loss = metrics_df[metrics_df['key'] == 'train_loss'].sort_values('step')
        val_loss = metrics_df[metrics_df['key'] == 'val_loss'].sort_values('step')
        
        if len(train_loss) > 0 and len(val_loss) > 0:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            axes[0].plot(train_loss['step'], train_loss['value'], label='Train Loss', marker='o')
            axes[0].plot(val_loss['step'], val_loss['value'], label='Val Loss', marker='o')
            axes[0].set_xlabel('Epoch')
            axes[0].set_ylabel('Loss')
            axes[0].set_title('Training vs Validation Loss')
            axes[0].legend()
            
            # Overfitting gap
            gap = val_loss['value'].values - train_loss['value'].values[:len(val_loss)]
            axes[1].plot(range(len(gap)), gap, marker='o', color='red')
            axes[1].axhline(0, color='black', linestyle='--')
            axes[1].set_xlabel('Epoch')
            axes[1].set_ylabel('Val Loss - Train Loss')
            axes[1].set_title('Overfitting Gap (positive = overfitting)')
            
            plt.tight_layout()
            plt.show()
            
            print(f"\nFinal train loss: {train_loss['value'].iloc[-1]:.4f}")
            print(f"Final val loss: {val_loss['value'].iloc[-1]:.4f}")
            print(f"Best val loss: {val_loss['value'].min():.4f}")
            print(f"Overfitting ratio: {val_loss['value'].iloc[-1] / train_loss['value'].iloc[-1]:.2f}x")
    
    conn.close()
else:
    print("No MLflow database found - skipping training analysis")

## 6. Baseline Comparisons

Compare the trained model against simple baselines:
1. Random recommendations
2. Most popular items
3. Category-based popularity
4. Content-based (visual similarity)

In [ ]:
from src.evaluation.metrics import RecommenderEvaluator, precision_at_k, recall_at_k, hit_rate_at_k, ndcg_at_k

# Prepare evaluation data
train_df, val_df, test_df = processor.prepare_datasets(temporal_split=True)

# Build ground truth for test users
test_ground_truth = {}
for user_idx, group in test_df.groupby('user_idx'):
    test_ground_truth[user_idx] = set(group['item_idx'].values)

# Get train items per user (to exclude from recommendations)
train_user_items = {}
for user_idx, group in train_df.groupby('user_idx'):
    train_user_items[user_idx] = set(group['item_idx'].values)

print(f"Test users: {len(test_ground_truth)}")
print(f"Avg test items per user: {np.mean([len(v) for v in test_ground_truth.values()]):.1f}")

In [ ]:
def evaluate_baseline(recommendations, ground_truth, name, k_values=[5, 10, 20, 50]):
    """Evaluate a baseline model."""
    results = {}
    for k in k_values:
        hits = []
        precisions = []
        recalls = []
        ndcgs = []
        
        for user_idx, recs in recommendations.items():
            if user_idx not in ground_truth or len(ground_truth[user_idx]) == 0:
                continue
            
            relevant = ground_truth[user_idx]
            hits.append(hit_rate_at_k(recs, relevant, k))
            precisions.append(precision_at_k(recs, relevant, k))
            recalls.append(recall_at_k(recs, relevant, k))
            ndcgs.append(ndcg_at_k(recs, relevant, k))
        
        results[f'hit_rate@{k}'] = np.mean(hits)
        results[f'precision@{k}'] = np.mean(precisions)
        results[f'recall@{k}'] = np.mean(recalls)
        results[f'ndcg@{k}'] = np.mean(ndcgs)
    
    return results

# Store all baseline results
baseline_results = {}

In [ ]:
# Baseline 1: Random
print("Evaluating Random baseline...")
all_items = list(range(len(items_df)))
random_recs = {}

for user_idx in test_ground_truth:
    exclude = train_user_items.get(user_idx, set())
    candidates = [i for i in all_items if i not in exclude]
    random_recs[user_idx] = list(np.random.choice(candidates, min(50, len(candidates)), replace=False))

baseline_results['Random'] = evaluate_baseline(random_recs, test_ground_truth, 'Random')
print(f"  hit_rate@10: {baseline_results['Random']['hit_rate@10']:.4f}")

In [ ]:
# Baseline 2: Most Popular
print("Evaluating Popularity baseline...")
item_popularity = train_df.groupby('item_idx').size().sort_values(ascending=False)
popular_items = item_popularity.index.tolist()

popularity_recs = {}
for user_idx in test_ground_truth:
    exclude = train_user_items.get(user_idx, set())
    recs = [i for i in popular_items if i not in exclude][:50]
    popularity_recs[user_idx] = recs

baseline_results['Popularity'] = evaluate_baseline(popularity_recs, test_ground_truth, 'Popularity')
print(f"  hit_rate@10: {baseline_results['Popularity']['hit_rate@10']:.4f}")

In [ ]:
# Baseline 3: Category-aware Popularity
print("Evaluating Category-Popularity baseline...")

# Get user's preferred categories from train data
user_categories = {}
for user_idx, group in train_df.groupby('user_idx'):
    item_idxs = group['item_idx'].values
    cats = items_df.set_index('item_idx').loc[item_idxs, 'category_idx'].value_counts()
    user_categories[user_idx] = cats.index.tolist()[:3]  # Top 3 categories

# Category -> popular items
cat_popular = {}
for cat_idx in items_df['category_idx'].unique():
    cat_items = items_df[items_df['category_idx'] == cat_idx]['item_idx'].values
    cat_pop = item_popularity[item_popularity.index.isin(cat_items)].index.tolist()
    cat_popular[cat_idx] = cat_pop

cat_pop_recs = {}
for user_idx in test_ground_truth:
    exclude = train_user_items.get(user_idx, set())
    user_cats = user_categories.get(user_idx, [])
    
    recs = []
    for cat in user_cats:
        for item in cat_popular.get(cat, []):
            if item not in exclude and item not in recs:
                recs.append(item)
            if len(recs) >= 50:
                break
        if len(recs) >= 50:
            break
    
    # Fill with global popular if needed
    if len(recs) < 50:
        for item in popular_items:
            if item not in exclude and item not in recs:
                recs.append(item)
            if len(recs) >= 50:
                break
    
    cat_pop_recs[user_idx] = recs

baseline_results['Category-Pop'] = evaluate_baseline(cat_pop_recs, test_ground_truth, 'Category-Pop')
print(f"  hit_rate@10: {baseline_results['Category-Pop']['hit_rate@10']:.4f}")

In [ ]:
# Baseline 4: Content-based (Visual Similarity)
if visual_embeddings is not None:
    print("Evaluating Content-based (Visual) baseline...")
    
    # Normalize embeddings
    visual_normed = visual_embeddings / (np.linalg.norm(visual_embeddings, axis=1, keepdims=True) + 1e-8)
    
    content_recs = {}
    for user_idx in list(test_ground_truth.keys())[:500]:  # Sample for speed
        train_items = list(train_user_items.get(user_idx, []))
        exclude = train_user_items.get(user_idx, set())
        
        if len(train_items) == 0:
            content_recs[user_idx] = popularity_recs[user_idx]
            continue
        
        # Average embedding of user's items
        valid_items = [i for i in train_items if i < len(visual_normed)]
        if len(valid_items) == 0:
            content_recs[user_idx] = popularity_recs[user_idx]
            continue
            
        user_emb = visual_normed[valid_items].mean(axis=0)
        
        # Compute similarities
        sims = np.dot(visual_normed, user_emb)
        
        # Get top items excluding train
        ranked = np.argsort(-sims)
        recs = [i for i in ranked if i not in exclude][:50]
        content_recs[user_idx] = recs
    
    baseline_results['Content-Visual'] = evaluate_baseline(content_recs, test_ground_truth, 'Content-Visual')
    print(f"  hit_rate@10: {baseline_results['Content-Visual']['hit_rate@10']:.4f}")

In [ ]:
# Compare all baselines
print("\n" + "=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

comparison_df = pd.DataFrame(baseline_results).T
comparison_df = comparison_df[['hit_rate@5', 'hit_rate@10', 'hit_rate@20', 'hit_rate@50', 
                               'precision@10', 'recall@10', 'ndcg@10']]

# Add your model results
your_model_results = {
    'hit_rate@5': 0.0081,
    'hit_rate@10': 0.0146,
    'hit_rate@20': 0.0227,
    'hit_rate@50': 0.0474,
    'precision@10': 0.0015,
    'recall@10': 0.0026,
    'ndcg@10': 0.0021
}
comparison_df.loc['TwoTower (Current)'] = your_model_results

print(comparison_df.round(4).to_string())

print("\n" + "=" * 60)
print("DIAGNOSIS:")
print("=" * 60)

best_baseline = comparison_df['hit_rate@10'].idxmax()
best_score = comparison_df['hit_rate@10'].max()
model_score = your_model_results['hit_rate@10']

if model_score < best_score:
    print(f"⚠️ Your model ({model_score:.4f}) is WORSE than {best_baseline} ({best_score:.4f})!")
    print("   This indicates a fundamental problem with the model.")
else:
    print(f"✓ Your model beats the baselines by {(model_score/best_score - 1)*100:.1f}%")

## 7. Root Cause Analysis

In [ ]:
print("\n" + "=" * 60)
print("ROOT CAUSE ANALYSIS")
print("=" * 60)

issues = []

# Check 1: Data Sparsity
if density < 0.1:
    issues.append({
        'issue': 'Extreme data sparsity',
        'detail': f'Only {density:.4f}% of user-item pairs observed',
        'fix': 'Use content-based features, regularization, or hybrid approaches'
    })

# Check 2: Cold Start
cold_pct = (user_counts <= 3).mean() * 100
if cold_pct > 30:
    issues.append({
        'issue': 'Cold start problem',
        'detail': f'{cold_pct:.1f}% of users have ≤3 interactions',
        'fix': 'Use content-based features for cold users, or use popularity fallback'
    })

# Check 3: Overfitting (based on training logs)
# val_loss went from 1.6 to 4.7 while train went from 1.6 to 0.76
issues.append({
    'issue': 'Severe overfitting',
    'detail': 'Val loss increased 3x (1.6→4.7) while train decreased (1.6→0.76)',
    'fix': 'Increase dropout, add L2 regularization, reduce model capacity, use early stopping'
})

# Check 4: Embedding dimension
issues.append({
    'issue': 'Small embedding dimension',
    'detail': 'Output dim=32 may be too small for 32K items',
    'fix': 'Try 64 or 128 dimensional embeddings'
})

# Check 5: Temporal split leakage check
train_users = set(train_df['user_idx'])
test_users = set(test_df['user_idx'])
overlap = len(train_users & test_users) / len(test_users) * 100
if overlap < 50:
    issues.append({
        'issue': 'Low user overlap between train/test',
        'detail': f'Only {overlap:.1f}% of test users appear in train',
        'fix': 'Use leave-one-out or user-stratified splitting'
    })

print("\nIdentified Issues:")
print("-" * 60)
for i, issue in enumerate(issues, 1):
    print(f"\n{i}. {issue['issue']}")
    print(f"   Detail: {issue['detail']}")
    print(f"   Fix: {issue['fix']}")

## 8. Recommendations for Improvement

In [ ]:
print("\n" + "=" * 60)
print("RECOMMENDATIONS FOR IMPROVEMENT")
print("=" * 60)

recommendations = """
QUICK WINS (Try these first):
─────────────────────────────
1. Increase regularization:
   - Set dropout to 0.4-0.5 (currently 0.2)
   - Add L2 weight decay: 1e-4 to 1e-3

2. Fix evaluation split:
   - Use leave-one-out evaluation (last item per user as test)
   - Ensure test users have training history

3. Use larger embedding dimension:
   - Try 64 or 128 instead of 32

MEDIUM EFFORT:
──────────────
4. Improve negative sampling:
   - Use popularity-weighted negatives
   - Increase num_negatives from 4 to 10+

5. Try simpler models first:
   - Matrix Factorization (SVD/ALS)
   - Implicit feedback models (BPR)

6. Better visual features:
   - Use CLIP for text+image embeddings
   - Fine-tune on your fashion data

ALTERNATIVE APPROACHES:
───────────────────────
7. Use established libraries:
   - Implicit (Python library for collaborative filtering)
   - LightFM (hybrid model)
   - RecBole (comprehensive RecSys framework)

8. Try Graph-based methods:
   - LightGCN
   - NGCF

9. Consider external APIs:
   - Recombee
   - AWS Personalize
   - Google Recommendations AI
"""

print(recommendations)

## 9. Quick Experiments

In [ ]:
# Try Implicit library (ALS/BPR) as comparison
try:
    import implicit
    from scipy.sparse import csr_matrix
    
    print("Testing Implicit library (ALS)...")
    
    # Build sparse matrix
    rows = train_df['user_idx'].values
    cols = train_df['item_idx'].values
    data = train_df['interaction_strength'].values
    
    user_item = csr_matrix((data, (rows, cols)), shape=(num_users, num_items))
    
    # Train ALS model
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64,
        regularization=0.1,
        iterations=15,
        random_state=42
    )
    als_model.fit(user_item)
    
    # Generate recommendations
    als_recs = {}
    for user_idx in list(test_ground_truth.keys())[:500]:
        if user_idx < user_item.shape[0]:
            exclude = train_user_items.get(user_idx, set())
            ids, scores = als_model.recommend(
                user_idx, 
                user_item[user_idx],
                N=50,
                filter_already_liked_items=True
            )
            als_recs[user_idx] = ids.tolist()
    
    baseline_results['ALS (Implicit)'] = evaluate_baseline(als_recs, test_ground_truth, 'ALS')
    print(f"  hit_rate@10: {baseline_results['ALS (Implicit)']['hit_rate@10']:.4f}")
    
except ImportError:
    print("Implicit library not installed. Install with: pip install implicit")

In [ ]:
# Try LightFM (hybrid model)
try:
    from lightfm import LightFM
    from lightfm.data import Dataset
    from lightfm.evaluation import precision_at_k as lfm_precision
    
    print("Testing LightFM (hybrid)...")
    
    # Build dataset
    dataset = Dataset()
    dataset.fit(
        users=train_df['user_idx'].unique(),
        items=items_df['item_idx'].unique()
    )
    
    interactions, _ = dataset.build_interactions(
        [(r['user_idx'], r['item_idx'], r['interaction_strength']) 
         for _, r in train_df.iterrows()]
    )
    
    # Train model
    lfm_model = LightFM(loss='warp', no_components=64, learning_rate=0.05)
    lfm_model.fit(interactions, epochs=10, num_threads=4, verbose=False)
    
    # Generate recommendations
    n_users, n_items = interactions.shape
    lfm_recs = {}
    
    for user_idx in list(test_ground_truth.keys())[:500]:
        if user_idx < n_users:
            scores = lfm_model.predict(user_idx, np.arange(n_items))
            exclude = train_user_items.get(user_idx, set())
            scores[list(exclude)] = -np.inf
            top_items = np.argsort(-scores)[:50]
            lfm_recs[user_idx] = top_items.tolist()
    
    baseline_results['LightFM'] = evaluate_baseline(lfm_recs, test_ground_truth, 'LightFM')
    print(f"  hit_rate@10: {baseline_results['LightFM']['hit_rate@10']:.4f}")
    
except ImportError:
    print("LightFM not installed. Install with: pip install lightfm")

In [ ]:
# Final comparison table
print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)

final_comparison = pd.DataFrame(baseline_results).T
final_comparison = final_comparison[['hit_rate@10', 'precision@10', 'recall@10', 'ndcg@10']]
final_comparison['TwoTower (Current)'] = [0.0146, 0.0015, 0.0026, 0.0021]
final_comparison = final_comparison.T

# Sort by hit_rate@10
final_comparison = final_comparison.sort_values('hit_rate@10', ascending=False).T

print(final_comparison.round(4).to_string())

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
best = final_comparison.T.idxmax()['hit_rate@10']
print(f"Best performing model: {best}")
print(f"Recommendation: Start with {best} as baseline, then improve from there.")

## 10. Next Steps

Based on this analysis, here are the recommended next steps:

1. **Fix the evaluation methodology** - ensure test users have training history
2. **Try simpler models** - ALS, BPR, or LightFM often outperform neural models on sparse data
3. **Add regularization** - the current model is severely overfitting
4. **Consider external APIs** - for quick wins, try Recombee or similar services